### Chunking for Data & AI Training 

---

## Table of Contents

1. [Introduction — Why Chunking Matters](#1)
2. [Setup & Sample Data](#2)
3. [Fixed-Size Chunking](#3)
4. [Sliding Window (Overlapping) Chunking](#4)
5. [Sentence-Based Chunking](#5)
6. [Paragraph / Section-Based Chunking](#6)
7. [Recursive Character Chunking](#7)
8. [Semantic Chunking](#8)
9. [Token-Based Chunking](#9)
10. [Document-Aware Chunking (Markdown / HTML)](#10)
11. [Agentic / Contextual Chunking](#11)
12. [Evaluation — Comparing Strategies](#12)
13. [Best Practices & Decision Guide](#13)
14. [Conclusion & Further Reading](#14)

---

<a id='1'></a>
## 1. Introduction — Why Chunking Matters

**Chunking** is the process of breaking large text documents into smaller, meaningful pieces ("chunks") before feeding them to AI models. It is a foundational step in:

- **Retrieval-Augmented Generation (RAG):** Embedding and retrieving relevant passages for LLM context windows.
- **Fine-Tuning Data Preparation:** Creating training samples that fit within model token limits.
- **Vector Database Indexing:** Storing semantically coherent units for similarity search.
- **Text Summarization Pipelines:** Breaking long documents into digestible pieces.

### Why does chunk quality matter?

| Problem | Consequence |
|---------|------------|
| Chunks too large | Exceeds token limits; irrelevant context dilutes retrieval quality |
| Chunks too small | Loses semantic coherence; increases noise in embeddings |
| Bad split points | Cuts mid-sentence/mid-idea, producing meaningless fragments |
| No overlap | Misses information at chunk boundaries |

This notebook implements **10+ chunking strategies from scratch**, compares them on real text, and gives you a decision framework to pick the right one for your use case.

<a id='2'></a>
## 2. Setup & Sample Data

In [ ]:
# ============================================================
# Install dependencies (uncomment if needed)
# ============================================================
# !pip install nltk tiktoken numpy scikit-learn beautifulsoup4 tabulate

In [ ]:
# ============================================================
# Core imports used throughout the notebook
# ============================================================
import re
import textwrap
from typing import List, Tuple, Optional

import numpy as np

# Pretty-print helper
def show_chunks(chunks: list, max_display: int = 5, max_chars: int = 120):
    """Display chunk list with index, length, and a preview."""
    print(f"Total chunks: {len(chunks)}")
    print(f"Avg length  : {np.mean([len(c) for c in chunks]):.0f} chars")
    print(f"Min / Max   : {min(len(c) for c in chunks)} / {max(len(c) for c in chunks)} chars")
    print("-" * 80)
    for i, chunk in enumerate(chunks[:max_display]):
        preview = chunk[:max_chars].replace('\n', ' ')
        if len(chunk) > max_chars:
            preview += "..."
        print(f"  [{i}] ({len(chunk):>5} chars) {preview}")
    if len(chunks) > max_display:
        print(f"  ... and {len(chunks) - max_display} more chunks")

In [ ]:
# ============================================================
# Sample document — an article about machine learning
# (We use a long, realistic text to make chunking meaningful)
# ============================================================

SAMPLE_TEXT = """# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Rather than being explicitly programmed to perform a task, these systems use algorithms to identify patterns in data and make predictions or decisions based on those patterns.

The field has grown enormously over the past decade, driven by increases in computational power, the availability of large datasets, and advances in algorithm design. Today, machine learning powers a wide range of applications, from recommendation systems on streaming platforms to autonomous vehicles navigating complex environments.

## Supervised Learning

Supervised learning is one of the most common paradigms in machine learning. In this approach, the algorithm is trained on a labeled dataset, meaning that each training example is paired with an output label. The algorithm learns to map inputs to outputs by finding patterns in the training data.

Common supervised learning algorithms include linear regression for continuous outputs, logistic regression for binary classification, decision trees that split data based on feature values, and support vector machines that find optimal hyperplanes separating classes. Neural networks, particularly deep learning models, have become increasingly popular due to their ability to learn complex, hierarchical representations of data.

The process of training a supervised model typically involves splitting the dataset into training, validation, and test sets. The model is trained on the training set, hyperparameters are tuned using the validation set, and final performance is evaluated on the held-out test set. Cross-validation is often used to get a more robust estimate of model performance.

## Unsupervised Learning

Unlike supervised learning, unsupervised learning deals with unlabeled data. The goal is to discover hidden patterns or intrinsic structures in the input data without explicit guidance on what to look for.

Clustering algorithms like K-means, DBSCAN, and hierarchical clustering group similar data points together. These are widely used in customer segmentation, anomaly detection, and image compression. Dimensionality reduction techniques like PCA (Principal Component Analysis) and t-SNE help visualize high-dimensional data and reduce computational costs.

Generative models, such as variational autoencoders (VAEs) and generative adversarial networks (GANs), learn to generate new data samples that resemble the training distribution. These models have found applications in image synthesis, data augmentation, and drug discovery.

## Reinforcement Learning

Reinforcement learning (RL) takes a fundamentally different approach. An agent learns to make decisions by interacting with an environment. The agent receives rewards or penalties based on its actions and aims to maximize the cumulative reward over time.

Key concepts in RL include the state space (all possible states of the environment), the action space (all possible actions the agent can take), the reward function (which provides feedback on actions), and the policy (the strategy the agent follows). The exploration-exploitation tradeoff is central to RL: the agent must balance exploring new actions to discover their effects with exploiting known actions that yield high rewards.

Deep reinforcement learning combines neural networks with RL, enabling agents to handle complex, high-dimensional state spaces. Notable successes include AlphaGo defeating world champion Go players, agents learning to play Atari games from raw pixels, and robotic control systems that learn dexterous manipulation tasks.

## Training Data Preparation

The quality of training data directly impacts model performance. Data preparation involves several critical steps:

Data collection and labeling form the foundation. High-quality labels are essential for supervised learning, and techniques like active learning can make the labeling process more efficient by focusing human effort on the most informative examples.

Data cleaning involves handling missing values, removing duplicates, correcting errors, and dealing with outliers. Feature engineering transforms raw data into features that better represent the underlying problem, often requiring domain expertise.

Data augmentation artificially increases the size and diversity of the training set. In computer vision, this might involve random rotations, flips, crops, and color adjustments. In natural language processing, techniques include synonym replacement, back-translation, and paraphrasing.

Text chunking is a critical preprocessing step when working with large documents for AI training. Breaking text into appropriately sized pieces ensures that each chunk fits within model context windows, maintains semantic coherence, and provides useful training signals.

## Evaluation and Metrics

Evaluating model performance requires choosing appropriate metrics. For classification tasks, common metrics include accuracy, precision, recall, F1-score, and area under the ROC curve (AUC-ROC). For regression tasks, metrics like mean squared error (MSE), mean absolute error (MAE), and R-squared are standard.

Beyond individual metrics, it is crucial to assess model fairness, robustness, and interpretability. Bias in training data can lead to discriminatory predictions, making fairness audits essential. Robustness testing ensures models perform reliably under distribution shifts and adversarial inputs. Interpretability techniques like SHAP values and attention visualization help stakeholders understand and trust model decisions.

## Conclusion

Machine learning continues to evolve rapidly, with new architectures, training techniques, and applications emerging regularly. Understanding the fundamentals—supervised learning, unsupervised learning, reinforcement learning, data preparation, and evaluation—provides a solid foundation for working with these powerful tools. The careful preparation of training data, including effective chunking strategies, plays a vital role in building successful AI systems.
"""

print(f"Document length: {len(SAMPLE_TEXT):,} characters")
print(f"Approx words  : {len(SAMPLE_TEXT.split()):,}")

---
<a id='3'></a>
## 3. Fixed-Size Chunking

The simplest strategy: slice the text into chunks of exactly `N` characters (or words).

**Pros:** Dead-simple, predictable size, fast.  
**Cons:** Cuts mid-word/mid-sentence, no semantic awareness.  
**Best for:** Quick prototyping, logs, or data where structure doesn't matter.

In [ ]:
# ============================================================
# 3a. Fixed-size character chunking
# ============================================================

def chunk_fixed_size(text: str, chunk_size: int = 500) -> List[str]:
    """
    Split text into fixed-size character chunks.
    
    Parameters
    ----------
    text : str
        Input document.
    chunk_size : int
        Number of characters per chunk.
    
    Returns
    -------
    List[str]
    """
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]


chunks_fixed = chunk_fixed_size(SAMPLE_TEXT, chunk_size=500)
show_chunks(chunks_fixed)

In [ ]:
# ============================================================
# 3b. Fixed-size word chunking
# ============================================================

def chunk_fixed_words(text: str, num_words: int = 100) -> List[str]:
    """
    Split text into chunks of a fixed number of words.
    """
    words = text.split()
    return [
        " ".join(words[i : i + num_words])
        for i in range(0, len(words), num_words)
    ]


chunks_words = chunk_fixed_words(SAMPLE_TEXT, num_words=80)
show_chunks(chunks_words)

### Notice the problem
Look at the chunk boundaries — you'll likely see sentences cut in half. This hurts embedding quality and retrieval accuracy.

---
<a id='4'></a>
## 4. Sliding Window (Overlapping) Chunking

Adds **overlap** between consecutive chunks so information at boundaries isn't lost.

**Pros:** Reduces boundary information loss; great for RAG.  
**Cons:** Increases total number of chunks (more storage/compute); still splits mid-sentence.  
**Best for:** RAG pipelines, embedding workflows.

In [ ]:
# ============================================================
# 4. Sliding window chunking with overlap
# ============================================================

def chunk_sliding_window(
    text: str,
    chunk_size: int = 500,
    overlap: int = 100,
) -> List[str]:
    """
    Sliding window chunking with configurable overlap.
    
    Parameters
    ----------
    text : str
    chunk_size : int
        Characters per chunk.
    overlap : int
        Number of overlapping characters between consecutive chunks.
    """
    assert overlap < chunk_size, "Overlap must be smaller than chunk size"
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(text), step):
        chunk = text[i : i + chunk_size]
        if chunk.strip():  # skip empty trailing chunks
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks


chunks_sliding = chunk_sliding_window(SAMPLE_TEXT, chunk_size=500, overlap=100)
show_chunks(chunks_sliding)

print(f"\nCompare: Fixed has {len(chunks_fixed)} chunks vs Sliding has {len(chunks_sliding)} chunks")

In [ ]:
# ============================================================
# Visualize the overlap between chunk 0 and chunk 1
# ============================================================

c0, c1 = chunks_sliding[0], chunks_sliding[1]
overlap_text = c0[-100:]  # last 100 chars of chunk 0

print("=== End of Chunk 0 ===")
print(repr(c0[-120:]))
print("\n=== Start of Chunk 1 ===")
print(repr(c1[:120]))
print("\n=== Overlap region ===")
print(repr(overlap_text))

---
<a id='5'></a>
## 5. Sentence-Based Chunking

Splits on **sentence boundaries** and groups sentences until a size limit is reached. This preserves semantic units.

**Pros:** Never cuts mid-sentence; respects natural language boundaries.  
**Cons:** Chunk sizes vary; needs a good sentence tokenizer.  
**Best for:** General-purpose text, articles, documentation.

In [ ]:
# ============================================================
# 5a. Regex-based sentence splitter (no external deps)
# ============================================================

def split_sentences_regex(text: str) -> List[str]:
    """Split text into sentences using regex."""
    # Split on period, question mark, exclamation mark followed by space or newline
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def chunk_by_sentences(
    text: str,
    max_chunk_size: int = 500,
    overlap_sentences: int = 1,
) -> List[str]:
    """
    Group sentences into chunks up to max_chunk_size characters,
    with optional sentence-level overlap.
    """
    sentences = split_sentences_regex(text)
    chunks = []
    current_chunk: List[str] = []
    current_length = 0
    
    for sent in sentences:
        sent_len = len(sent)
        
        # If adding this sentence would exceed the limit, finalize current chunk
        if current_length + sent_len > max_chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            # Keep the last N sentences as overlap
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences else []
            current_length = sum(len(s) for s in current_chunk)
        
        current_chunk.append(sent)
        current_length += sent_len
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks


chunks_sentence = chunk_by_sentences(SAMPLE_TEXT, max_chunk_size=500, overlap_sentences=1)
show_chunks(chunks_sentence, max_display=6)

In [ ]:
# ============================================================
# 5b. NLTK sentence tokenizer (more accurate)
# ============================================================

import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize


def chunk_by_sentences_nltk(
    text: str,
    max_chunk_size: int = 500,
    overlap_sentences: int = 1,
) -> List[str]:
    """Sentence-based chunking using NLTK's sentence tokenizer."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk: List[str] = []
    current_length = 0
    
    for sent in sentences:
        sent_len = len(sent)
        if current_length + sent_len > max_chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences else []
            current_length = sum(len(s) for s in current_chunk)
        
        current_chunk.append(sent)
        current_length += sent_len
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks


chunks_nltk = chunk_by_sentences_nltk(SAMPLE_TEXT, max_chunk_size=500, overlap_sentences=1)
show_chunks(chunks_nltk, max_display=6)

print(f"\nRegex-based: {len(chunks_sentence)} chunks | NLTK-based: {len(chunks_nltk)} chunks")

---
<a id='6'></a>
## 6. Paragraph / Section-Based Chunking

Splits on **double newlines** (paragraph breaks) or **headings**, preserving the document's structural boundaries.

**Pros:** Preserves author-intended logical groupings.  
**Cons:** Paragraph sizes can vary hugely; may need secondary splitting for long paragraphs.  
**Best for:** Structured documents, reports, markdown docs.

In [ ]:
# ============================================================
# 6a. Paragraph chunking
# ============================================================

def chunk_by_paragraphs(
    text: str,
    max_chunk_size: int = 800,
) -> List[str]:
    """
    Split on paragraph boundaries (double newlines).
    Merge short paragraphs together; split long ones by sentence.
    """
    paragraphs = re.split(r'\n\s*\n', text.strip())
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        # If the paragraph alone exceeds the limit, split it by sentence
        if len(para) > max_chunk_size:
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""
            # Sub-split long paragraphs
            sub_chunks = chunk_by_sentences(para, max_chunk_size=max_chunk_size, overlap_sentences=0)
            chunks.extend(sub_chunks)
        elif len(current_chunk) + len(para) + 2 > max_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk = current_chunk + "\n\n" + para if current_chunk else para
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks


chunks_para = chunk_by_paragraphs(SAMPLE_TEXT, max_chunk_size=800)
show_chunks(chunks_para, max_display=6)

In [ ]:
# ============================================================
# 6b. Section-based chunking (split on Markdown headings)
# ============================================================

def chunk_by_sections(text: str) -> List[dict]:
    """
    Split a Markdown document by heading boundaries.
    Returns dicts with 'heading' and 'content' keys.
    """
    # Match lines starting with one or more '#'
    pattern = r'^(#{1,6})\s+(.+)$'
    lines = text.split('\n')
    sections = []
    current_heading = "(Preamble)"
    current_level = 0
    current_lines = []
    
    for line in lines:
        match = re.match(pattern, line)
        if match:
            # Save previous section
            if current_lines:
                content = '\n'.join(current_lines).strip()
                if content:
                    sections.append({
                        'heading': current_heading,
                        'level': current_level,
                        'content': content,
                    })
            current_level = len(match.group(1))
            current_heading = match.group(2)
            current_lines = []
        else:
            current_lines.append(line)
    
    # Last section
    if current_lines:
        content = '\n'.join(current_lines).strip()
        if content:
            sections.append({
                'heading': current_heading,
                'level': current_level,
                'content': content,
            })
    
    return sections


sections = chunk_by_sections(SAMPLE_TEXT)
for sec in sections:
    print(f"  [L{sec['level']}] {sec['heading']:40s} ({len(sec['content']):>5} chars)")

---
<a id='7'></a>
## 7. Recursive Character Chunking

This is the strategy used by **LangChain's `RecursiveCharacterTextSplitter`**. It tries a hierarchy of separators (e.g., `\n\n`, `\n`, `. `, ` `) and falls back to the next one when chunks are still too large.

**Pros:** Adaptively picks the best split point; balances structure and size.  
**Cons:** Slightly more complex logic.  
**Best for:** General-purpose chunking (often the best default choice).

In [ ]:
# ============================================================
# 7. Recursive character text splitter (from scratch)
# ============================================================

def chunk_recursive(
    text: str,
    chunk_size: int = 500,
    chunk_overlap: int = 50,
    separators: Optional[List[str]] = None,
) -> List[str]:
    """
    Recursively split text using a hierarchy of separators.
    Mimics LangChain's RecursiveCharacterTextSplitter.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", ", ", " ", ""]
    
    final_chunks: List[str] = []
    
    def _split(text: str, seps: List[str]) -> List[str]:
        """Try to split text with the first separator; recurse on remaining."""
        if not text.strip():
            return []
        
        # Base case: text fits in one chunk
        if len(text) <= chunk_size:
            return [text]
        
        # No more separators — hard split
        if not seps:
            return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size - chunk_overlap)]
        
        sep = seps[0]
        remaining_seps = seps[1:]
        
        if sep == "":
            # Character-level split
            parts = list(text)
        else:
            parts = text.split(sep)
        
        chunks = []
        current = ""
        
        for part in parts:
            candidate = current + sep + part if current else part
            
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    # If current chunk is still too big, recurse with finer separators
                    if len(current) > chunk_size:
                        chunks.extend(_split(current, remaining_seps))
                    else:
                        chunks.append(current)
                
                # Start a new chunk; if part itself is too big, recurse
                if len(part) > chunk_size:
                    chunks.extend(_split(part, remaining_seps))
                    current = ""
                else:
                    current = part
        
        if current:
            if len(current) > chunk_size:
                chunks.extend(_split(current, remaining_seps))
            else:
                chunks.append(current)
        
        return chunks
    
    raw_chunks = _split(text, separators)
    
    # Apply overlap by prepending tail of previous chunk
    if chunk_overlap > 0 and len(raw_chunks) > 1:
        overlapped = [raw_chunks[0]]
        for i in range(1, len(raw_chunks)):
            prev_tail = raw_chunks[i-1][-chunk_overlap:]
            overlapped.append(prev_tail + raw_chunks[i])
        return overlapped
    
    return raw_chunks


chunks_recursive = chunk_recursive(SAMPLE_TEXT, chunk_size=500, chunk_overlap=50)
show_chunks(chunks_recursive, max_display=6)

---
<a id='8'></a>
## 8. Semantic Chunking

Instead of splitting on character counts, **semantic chunking** groups sentences by their *meaning*. We compute sentence embeddings and split where the semantic similarity between consecutive sentences drops significantly.

**Pros:** Produces highly coherent chunks aligned with topic shifts.  
**Cons:** Requires an embedding model; slower; chunk sizes vary widely.  
**Best for:** High-quality RAG, knowledge bases, question answering.

In [ ]:
# ============================================================
# 8. Semantic chunking using cosine similarity of embeddings
# ============================================================
# We simulate embeddings using TF-IDF for a zero-dependency demo.
# In production, replace with OpenAI/Cohere/HuggingFace embeddings.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def chunk_semantic(
    text: str,
    similarity_threshold: float = 0.15,
    min_chunk_sentences: int = 2,
) -> List[str]:
    """
    Semantic chunking: split where consecutive-sentence similarity drops.
    
    Parameters
    ----------
    text : str
    similarity_threshold : float
        Split when similarity between consecutive sentences falls below this.
    min_chunk_sentences : int
        Minimum sentences per chunk to avoid tiny fragments.
    """
    sentences = split_sentences_regex(text)
    
    if len(sentences) <= 2:
        return [text]
    
    # Compute TF-IDF embeddings for each sentence
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Compute cosine similarity between consecutive sentences
    similarities = []
    for i in range(len(sentences) - 1):
        sim = cosine_similarity(tfidf_matrix[i:i+1], tfidf_matrix[i+1:i+2])[0][0]
        similarities.append(sim)
    
    # Find split points where similarity drops below threshold
    split_indices = []
    for i, sim in enumerate(similarities):
        if sim < similarity_threshold and (i + 1) >= min_chunk_sentences:
            split_indices.append(i + 1)
    
    # Build chunks
    chunks = []
    start = 0
    for idx in split_indices:
        chunk_text = " ".join(sentences[start:idx])
        if chunk_text.strip():
            chunks.append(chunk_text)
        start = idx
    
    # Last chunk
    remaining = " ".join(sentences[start:])
    if remaining.strip():
        chunks.append(remaining)
    
    return chunks, similarities  # Return similarities for visualization


chunks_semantic, sim_scores = chunk_semantic(SAMPLE_TEXT, similarity_threshold=0.12)
show_chunks(chunks_semantic, max_display=6)

In [ ]:
# ============================================================
# Visualize semantic similarity between consecutive sentences
# ============================================================

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(sim_scores)), sim_scores, color='steelblue', alpha=0.7)
ax.axhline(y=0.12, color='red', linestyle='--', label='Threshold (0.12)')
ax.set_xlabel('Sentence Pair Index')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Consecutive Sentence Similarity — Split where bars fall below the red line')
ax.legend()
plt.tight_layout()
plt.savefig('semantic_similarity.png', dpi=100)
plt.show()
print("Plot saved to semantic_similarity.png")

---
<a id='9'></a>
## 9. Token-Based Chunking

LLMs count **tokens**, not characters. Token-based chunking ensures each chunk fits a model's context window exactly.

**Pros:** Directly maps to model limits; no wasted context.  
**Cons:** Token counts are model-specific; still needs sentence awareness.  
**Best for:** Fine-tuning data prep, prompt engineering, token-budget management.

In [ ]:
# ============================================================
# 9. Token-based chunking with tiktoken (OpenAI tokenizer)
# ============================================================

import tiktoken


def chunk_by_tokens(
    text: str,
    max_tokens: int = 200,
    overlap_tokens: int = 20,
    encoding_name: str = "cl100k_base",  # GPT-4 / GPT-3.5 tokenizer
) -> List[str]:
    """
    Split text into chunks of at most max_tokens tokens.
    Uses sentence boundaries to avoid mid-sentence splits.
    """
    enc = tiktoken.get_encoding(encoding_name)
    sentences = split_sentences_regex(text)
    
    chunks = []
    current_sentences = []
    current_tokens = 0
    
    for sent in sentences:
        sent_tokens = len(enc.encode(sent))
        
        if current_tokens + sent_tokens > max_tokens and current_sentences:
            chunk_text = " ".join(current_sentences)
            chunks.append(chunk_text)
            
            # Overlap: keep last sentences up to overlap_tokens
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_sentences):
                s_tokens = len(enc.encode(s))
                if overlap_count + s_tokens > overlap_tokens:
                    break
                overlap_sents.insert(0, s)
                overlap_count += s_tokens
            
            current_sentences = overlap_sents
            current_tokens = overlap_count
        
        current_sentences.append(sent)
        current_tokens += sent_tokens
    
    if current_sentences:
        chunks.append(" ".join(current_sentences))
    
    return chunks


chunks_token = chunk_by_tokens(SAMPLE_TEXT, max_tokens=200, overlap_tokens=20)
show_chunks(chunks_token, max_display=6)

# Show actual token counts
enc = tiktoken.get_encoding("cl100k_base")
token_counts = [len(enc.encode(c)) for c in chunks_token]
print(f"\nToken counts per chunk: {token_counts}")

---
<a id='10'></a>
## 10. Document-Aware Chunking (Markdown / HTML)

For structured documents, we can use the **markup itself** (headings, tags, sections) to guide chunking.

**Pros:** Leverages existing structure; produces topically focused chunks.  
**Cons:** Requires structured input; different logic per format.  
**Best for:** Documentation, wikis, web scraping, knowledge bases.

In [ ]:
# ============================================================
# 10a. Markdown-aware chunker
# ============================================================

def chunk_markdown(
    text: str,
    max_chunk_size: int = 800,
    include_heading_in_chunk: bool = True,
) -> List[str]:
    """
    Chunk Markdown by headings, keeping each section as a chunk.
    Long sections are sub-split by paragraph.
    """
    sections = chunk_by_sections(text)
    chunks = []
    
    for sec in sections:
        heading = sec['heading']
        content = sec['content']
        level = sec['level']
        
        if include_heading_in_chunk and heading != "(Preamble)":
            prefix = f"{'#' * level} {heading}\n\n"
        else:
            prefix = ""
        
        full_content = prefix + content
        
        if len(full_content) <= max_chunk_size:
            chunks.append(full_content)
        else:
            # Sub-split long sections by paragraph, prepending heading
            sub_chunks = chunk_by_paragraphs(content, max_chunk_size=max_chunk_size - len(prefix))
            for sc in sub_chunks:
                chunks.append(prefix + sc)
    
    return chunks


chunks_md = chunk_markdown(SAMPLE_TEXT, max_chunk_size=800)
show_chunks(chunks_md, max_display=8, max_chars=100)

In [ ]:
# ============================================================
# 10b. HTML-aware chunker
# ============================================================

from bs4 import BeautifulSoup

SAMPLE_HTML = """
<html><body>
<h1>Guide to Neural Networks</h1>
<p>Neural networks are computing systems inspired by biological neural networks.</p>
<p>They consist of layers of interconnected nodes that process information.</p>

<h2>Architecture</h2>
<p>A typical neural network has an input layer, one or more hidden layers, and an output layer.
Each connection between nodes has a weight that is adjusted during training.</p>
<p>Common architectures include feedforward networks, convolutional neural networks (CNNs),
and recurrent neural networks (RNNs). Transformers have become dominant in NLP tasks.</p>

<h2>Training Process</h2>
<p>Training involves forward propagation (computing predictions), loss calculation
(measuring error), and backpropagation (updating weights to minimize loss).</p>
<p>Optimization algorithms like SGD, Adam, and RMSprop control how weights are updated.
Learning rate scheduling and regularization techniques help prevent overfitting.</p>

<h2>Applications</h2>
<ul>
  <li>Computer vision: image classification, object detection</li>
  <li>Natural language processing: translation, summarization</li>
  <li>Generative AI: text generation, image synthesis</li>
</ul>
</body></html>
"""


def chunk_html(html: str, heading_tags: tuple = ('h1', 'h2', 'h3')) -> List[dict]:
    """Chunk HTML by heading tags, extracting text under each heading."""
    soup = BeautifulSoup(html, 'html.parser')
    chunks = []
    current_heading = "(Preamble)"
    current_content = []
    
    for element in soup.body.children if soup.body else []:
        if element.name in heading_tags:
            # Save previous section
            if current_content:
                chunks.append({
                    'heading': current_heading,
                    'content': ' '.join(current_content),
                })
            current_heading = element.get_text(strip=True)
            current_content = []
        elif element.name and element.get_text(strip=True):
            current_content.append(element.get_text(strip=True))
    
    if current_content:
        chunks.append({'heading': current_heading, 'content': ' '.join(current_content)})
    
    return chunks


html_chunks = chunk_html(SAMPLE_HTML)
for hc in html_chunks:
    print(f"  [{hc['heading']}] ({len(hc['content'])} chars)")
    print(f"    {hc['content'][:100]}...\n")

---
<a id='11'></a>
## 11. Agentic / Contextual Chunking

A more advanced technique: **prepend contextual metadata** to each chunk so that the chunk carries its own provenance. This helps retrieval systems understand where the chunk fits in the larger document.

**Pros:** Dramatically improves retrieval relevance; self-contained chunks.  
**Cons:** Increases chunk size; requires metadata extraction.  
**Best for:** Enterprise RAG, multi-document QA, production knowledge bases.

In [ ]:
# ============================================================
# 11. Contextual chunking — enrich chunks with metadata
# ============================================================

def chunk_contextual(
    text: str,
    doc_title: str = "Machine Learning Overview",
    doc_source: str = "internal_wiki",
    chunk_size: int = 500,
) -> List[dict]:
    """
    Create contextually enriched chunks that carry document metadata,
    section hierarchy, and position information.
    """
    sections = chunk_by_sections(text)
    enriched_chunks = []
    chunk_id = 0
    
    for sec_idx, sec in enumerate(sections):
        heading = sec['heading']
        content = sec['content']
        
        # Sub-chunk the content if needed
        if len(content) > chunk_size:
            sub_chunks = chunk_by_sentences(content, max_chunk_size=chunk_size, overlap_sentences=0)
        else:
            sub_chunks = [content]
        
        for sub_idx, sub_chunk in enumerate(sub_chunks):
            # Build the contextual prefix
            context_prefix = (
                f"Document: {doc_title}\n"
                f"Source: {doc_source}\n"
                f"Section: {heading}\n"
                f"Chunk: {chunk_id} (part {sub_idx+1}/{len(sub_chunks)} of section {sec_idx+1})\n"
                f"---\n"
            )
            
            enriched_chunks.append({
                'chunk_id': chunk_id,
                'section': heading,
                'section_index': sec_idx,
                'sub_index': sub_idx,
                'doc_title': doc_title,
                'source': doc_source,
                'content': sub_chunk,
                'content_with_context': context_prefix + sub_chunk,
                'char_count': len(sub_chunk),
            })
            chunk_id += 1
    
    return enriched_chunks


contextual_chunks = chunk_contextual(SAMPLE_TEXT, chunk_size=500)
print(f"Total enriched chunks: {len(contextual_chunks)}\n")

# Show one example
example = contextual_chunks[3]
print("=== Example Enriched Chunk ===")
print(f"Chunk ID   : {example['chunk_id']}")
print(f"Section    : {example['section']}")
print(f"Char count : {example['char_count']}")
print(f"\n--- Content with Context ---")
print(example['content_with_context'][:500])

---
<a id='12'></a>
## 12. Evaluation — Comparing Strategies

Let's build a **comparison framework** that measures each strategy across key dimensions:

| Metric | What it measures |
|--------|------------------|
| **Chunk count** | Total number of chunks |
| **Size consistency** | Standard deviation of chunk sizes |
| **Boundary quality** | % of chunks that start/end at sentence boundaries |
| **Coverage** | % of original text preserved in chunks |
| **Avg intra-chunk coherence** | Average self-similarity within each chunk |

In [ ]:
# ============================================================
# 12. Evaluate and compare all strategies
# ============================================================

def evaluate_chunks(chunks: List[str], original_text: str) -> dict:
    """Compute evaluation metrics for a set of chunks."""
    sizes = [len(c) for c in chunks]
    
    # Boundary quality: does the chunk start with a capital or heading?
    # and end with a sentence-terminal punctuation?
    good_starts = sum(1 for c in chunks if c.strip() and (c.strip()[0].isupper() or c.strip()[0] == '#'))
    good_ends = sum(1 for c in chunks if c.strip() and c.strip()[-1] in '.!?\n')
    boundary_quality = (good_starts + good_ends) / (2 * len(chunks)) * 100
    
    # Coverage: how much of the original text is represented?
    combined = " ".join(chunks)
    original_words = set(original_text.lower().split())
    chunk_words = set(combined.lower().split())
    coverage = len(original_words & chunk_words) / len(original_words) * 100
    
    return {
        'num_chunks': len(chunks),
        'avg_size': np.mean(sizes),
        'std_size': np.std(sizes),
        'min_size': min(sizes),
        'max_size': max(sizes),
        'boundary_quality': boundary_quality,
        'coverage': coverage,
    }


# Collect all strategies
strategies = {
    'Fixed (500 chars)':    chunks_fixed,
    'Sliding Window':       chunks_sliding,
    'Sentence-based':       chunks_sentence,
    'NLTK Sentence':        chunks_nltk,
    'Paragraph':            chunks_para,
    'Recursive':            chunks_recursive,
    'Semantic':             chunks_semantic,
    'Token-based':          chunks_token,
    'Markdown-aware':       chunks_md,
}

# Evaluate
results = {}
for name, chunks in strategies.items():
    results[name] = evaluate_chunks(chunks, SAMPLE_TEXT)

# Display as table
from tabulate import tabulate

table_data = []
for name, metrics in results.items():
    table_data.append([
        name,
        metrics['num_chunks'],
        f"{metrics['avg_size']:.0f}",
        f"{metrics['std_size']:.0f}",
        f"{metrics['min_size']}-{metrics['max_size']}",
        f"{metrics['boundary_quality']:.0f}%",
        f"{metrics['coverage']:.0f}%",
    ])

print(tabulate(
    table_data,
    headers=['Strategy', 'Chunks', 'Avg Size', 'Std Dev', 'Range', 'Boundary', 'Coverage'],
    tablefmt='grid',
))

In [ ]:
# ============================================================
# Visualization: Compare chunk size distributions
# ============================================================

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Chunk Size Distributions Across Strategies', fontsize=16, fontweight='bold')

for ax, (name, chunks) in zip(axes.flatten(), strategies.items()):
    sizes = [len(c) for c in chunks]
    ax.hist(sizes, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Chunk Size (chars)')
    ax.axvline(np.mean(sizes), color='red', linestyle='--', linewidth=1, label=f'Mean: {np.mean(sizes):.0f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('chunk_distributions.png', dpi=100)
plt.show()
print("Plot saved to chunk_distributions.png")

---
<a id='13'></a>
## 13. Best Practices & Decision Guide

### Choosing the Right Strategy

```
                         ┌─────────────────────────────────────┐
                         │  Is the document structured?        │
                         │  (Markdown, HTML, LaTeX, etc.)      │
                         └──────────────┬──────────────────────┘
                                        │
                           ┌────────────┴─────────────┐
                           │ YES                       │ NO
                           ▼                           ▼
                  Document-Aware Chunking     Is semantic coherence
                  (Section 10)                critical for your task?
                                                       │
                                          ┌────────────┴─────────────┐
                                          │ YES                      │ NO
                                          ▼                          ▼
                                   Semantic Chunking        Do you need exact
                                   (Section 8)              token budgeting?
                                                                    │
                                                       ┌────────────┴───────────┐
                                                       │ YES                    │ NO
                                                       ▼                        ▼
                                                Token-based              Recursive Character
                                                (Section 9)              (Section 7) ← DEFAULT
```

### Key Parameters to Tune

| Parameter | Guidance |
|-----------|----------|
| **Chunk size** | 256-512 tokens for retrieval; 1024+ for summarization |
| **Overlap** | 10-20% of chunk size is a good starting point |
| **Similarity threshold** | Tune on your domain; 0.1-0.3 for TF-IDF, 0.5-0.8 for dense embeddings |

### Common Pitfalls

1. **One-size-fits-all thinking** — Different tasks need different chunk sizes. Retrieval benefits from smaller chunks; summarization works better with larger ones.
2. **Ignoring overlap** — Without overlap, you lose context at boundaries. Always use at least some overlap for RAG.
3. **Not evaluating** — Always benchmark your chunking strategy against your actual downstream task.
4. **Forgetting metadata** — Contextual enrichment (Section 11) can dramatically improve retrieval quality.
5. **Tokenizer mismatch** — Use the same tokenizer as your target model for token-based chunking.

---
<a id='14'></a>
## 14. Conclusion & Further Reading

### What We Covered

This notebook implemented **10 chunking strategies** from scratch:

1. **Fixed-size** — Simple character/word slicing
2. **Sliding window** — Fixed-size with overlap
3. **Sentence-based** — Group sentences to a size limit (regex + NLTK)
4. **Paragraph-based** — Split on paragraph boundaries
5. **Section-based** — Split on document headings
6. **Recursive character** — Hierarchical separator fallback
7. **Semantic** — Split on topic shifts using embeddings
8. **Token-based** — Chunk by token count for LLM limits
9. **Document-aware** — Markdown & HTML structure-based
10. **Contextual/Agentic** — Metadata-enriched chunks

### Key Takeaways

- **Recursive character splitting** is the best default for most use cases.
- **Semantic chunking** gives the highest quality but requires embeddings.
- **Token-based chunking** is essential when preparing fine-tuning data.
- **Contextual enrichment** is a simple add-on that dramatically improves retrieval.
- Always **evaluate** your chunking against your downstream task.

### Further Reading

- [LangChain Text Splitters Documentation](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [LlamaIndex Node Parsers](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/)
- [Anthropic RAG Guide](https://docs.anthropic.com/en/docs/build-with-claude/retrieval-augmented-generation)
- [Pinecone Chunking Strategies](https://www.pinecone.io/learn/chunking-strategies/)
- Greg Kamradt: "5 Levels of Text Splitting" (YouTube)

---

*Happy chunking!